# **Unificación de shapefiles o vectores geoespaciales generados por tramos**

Este script permite unir múltiples archivos vectoriales (`.shp`) contenidos en una carpeta en un único archivo consolidado. Es especialmente útil cuando se han generado resultados por tramos (segmentos) y se desea contar con una **capa única por categoría de análisis**.

> 🛰️ **En este caso**, se utiliza para unir los vectores generados tras la detección de:
> - 🛣️ **Vías**: tramos segmentados y reconstruidos en pasos anteriores.
> - 🚗 **Automóviles**: objetos detectados mediante segmentación con LangSAM.
> - ⚠️ **Irregularidades**: patrones extraídos con otros modelos (por ejemplo, baches, obstáculos, sombreado).

### 🎯 Objetivo del proceso:
Consolidar todos los vectores asociados a un mismo tipo de entidad (vía, vehículo, irregularidad, etc.) detectados por tramos, en un único shapefile o geojson por troncal, facilitando la visualización y el análisis espacial.

### ✅ Funcionalidades principales:

- 📂 **Carga automatizada**: escanea una carpeta y detecta todos los archivos `.shp` para unir.
- 🧩 **Unificación geoespacial**: concatena los shapefiles en un único `GeoDataFrame`.
- 🏷️ **Columna de origen**: agrega una columna con el nombre del archivo de origen para trazabilidad y auditoría.
- 💾 **Exportación dual**:
  - `Shapefile (.shp)` para uso clásico en SIG.
  - `GeoJSON (.geojson)` para aplicaciones web o APIs geoespaciales.
- 🔁 **Procesamiento iterativo**: puede aplicarse a una o varias troncales automáticamente.

### 🛠️ Variables clave:

- `carpeta_entrada`: carpeta donde están los shapefiles a unir (por ejemplo, resultados de tramos).
- `output_shapefile`: ruta del archivo de salida consolidado.
- `troncales`: lista de identificadores (por ejemplo, nombres de vías) a procesar.
- `root`: ruta base común donde se almacenan los resultados por troncal y categoría.

### 🧠 Casos de uso típicos:

- Consolidar la detección de **vehículos** sobre una vía completa.
- Unificar los tramos de **vías reconstruidas** desde imágenes recortadas.
- Agrupar detecciones de **anomalías o irregularidades** sobre corredores viales.
- Preparar salidas limpias y organizadas para informes, dashboards o sistemas SIG.

Este paso final es esencial para entregar datos bien estructurados y fácilmente consumibles por usuarios técnicos o no técnicos, cerrando el ciclo completo desde segmentación por tramos hasta representación unificada.

In [ ]:
import geopandas as gpd
import os,glob
import pandas as pd


def unir_shapefiles(carpeta_entrada, output_shapefile):
    """
    Une todos los shapefiles de una carpeta y exporta un solo shapefile y un geojson.
    
    Parámetros:
    - carpeta_entrada: Ruta de la carpeta donde están los shapefiles.
    - output_shapefile: Ruta del archivo de salida.
    
    Retorna:
    - Rutas de los archivos generados.
    """
    # Buscar todos los archivos .shp en la carpeta
    shapefiles = [os.path.join(carpeta_entrada, f) for f in os.listdir(carpeta_entrada) if f.endswith(".shp")]
    
    if not shapefiles:
        raise FileNotFoundError("No se encontraron archivos .shp en la carpeta.")

    # Leer y unir los shapefiles con columna de origen
    gdfs = []
    for shp in shapefiles:
        print(shp)
        gdf = gpd.read_file(shp)
        gdf["origen"] = os.path.basename(shp)  # Agregar columna con el nombre del archivo
        gdfs.append(gdf)
    
    merged_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)

    # Guardar el shapefile y geojson
    merged_gdf.to_file(output_shapefile, driver="ESRI Shapefile")
    
    output_geojson = output_shapefile.replace('.shp','.geojson')
    merged_gdf.to_file(output_geojson, driver="GeoJSON")

    return output_shapefile, output_geojson


Unir todos los tramos de una troncal (vías o automóviles), el resultado sera **un shp por troncal con todos los vectores consolidados en un solo archivo**

In [ ]:
troncales=['Troncal5','Troncal2','Troncal3','Troncal4','Troncal1','Troncal9']

root='./data/MASKS_VIAS_FILL_Z19_BB22'#Unir todos los tramos de vía
root='./data/IMAGES\SAMprompts_ONLY_ROADS_FILL_z21_ExtBB22\cars, buses and trucks'#Unr todos los tramos de vehículos


for troncal in troncales:
    carpeta_entrada = os.path.join(root, troncal, 'bt0.37_tt0.37')  # Carpeta de entrada
    output_shapefile = os.path.join(root, troncal + '_cars.shp')  # Archivo de salida
    geojson_salida = output_shapefile.replace('.shp', '.geojson')  # Archivo GeoJSON

    # Verificar si los archivos ya existen
    if os.path.exists(output_shapefile) and os.path.exists(geojson_salida):
        print(f"Archivos ya existen para {troncal}, saltando...")
        continue

    shp_salida, geojson_salida = unir_shapefiles(carpeta_entrada, output_shapefile)
    print(f"Shapefile guardado en: {shp_salida}")
    print(f"GeoJSON guardado en: {geojson_salida}")


Unir los shapes por troncal para que quede **un solo shp con todas las troncales**

In [ ]:
root='./data/IMAGES/SAMprompts_ONLY_ROADS_FILL_z21_ExtBB22/cars, buses and trucks'#Unr todos los tramos de carros
root='./data/MASKS_VIAS_FILL_Z19_BB22'

files_shp=glob.glob(os.path.join(root,'*hole0.shp'))
files_json=glob.glob(os.path.join(root,'*hole0.geojson'))

files_shp=glob.glob(os.path.join(root,'*.shp'))
files_json=glob.glob(os.path.join(root,'*.geojson'))

files=files_shp+files_json
files

['D:\\CORREDORES_V_DATASET\\MASKS_VIAS_FILL_Z19_BB22\\Troncal1.shp',
 'D:\\CORREDORES_V_DATASET\\MASKS_VIAS_FILL_Z19_BB22\\Troncal2.shp',
 'D:\\CORREDORES_V_DATASET\\MASKS_VIAS_FILL_Z19_BB22\\Troncal3.shp',
 'D:\\CORREDORES_V_DATASET\\MASKS_VIAS_FILL_Z19_BB22\\Troncal4.shp',
 'D:\\CORREDORES_V_DATASET\\MASKS_VIAS_FILL_Z19_BB22\\Troncal5.shp',
 'D:\\CORREDORES_V_DATASET\\MASKS_VIAS_FILL_Z19_BB22\\Troncal9.shp',
 'D:\\CORREDORES_V_DATASET\\MASKS_VIAS_FILL_Z19_BB22\\Troncal1.geojson',
 'D:\\CORREDORES_V_DATASET\\MASKS_VIAS_FILL_Z19_BB22\\Troncal2.geojson',
 'D:\\CORREDORES_V_DATASET\\MASKS_VIAS_FILL_Z19_BB22\\Troncal3.geojson',
 'D:\\CORREDORES_V_DATASET\\MASKS_VIAS_FILL_Z19_BB22\\Troncal4.geojson',
 'D:\\CORREDORES_V_DATASET\\MASKS_VIAS_FILL_Z19_BB22\\Troncal5.geojson',
 'D:\\CORREDORES_V_DATASET\\MASKS_VIAS_FILL_Z19_BB22\\Troncal9.geojson']

In [ ]:
import geopandas as gpd
import os

# Lista de archivos a unir VIAS
files_shp=glob.glob(os.path.join(root,'*.shp'))
files_json=glob.glob(os.path.join(root,'*.geojson'))

#FIltered cars
files_shp=glob.glob(os.path.join(root,'*hole0.shp'))
#files_json=glob.glob(os.path.join(root,'*hole0.geojson'))
files=files_shp+files_json

# Lista para almacenar los GeoDataFrames
gdfs = []

for file in files:
    if os.path.exists(file):
        gdf = gpd.read_file(file)
        gdfs.append(gdf)
        print(f"Archivo cargado: {file}")
    else:
        print(f"Archivo no encontrado: {file}")

# Unir todos los GeoDataFrames
if gdfs:
    merged_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))
    
    # Definir el CRS si no está definido
    if merged_gdf.crs is None:
        merged_gdf.set_crs(epsg=4326, inplace=True)
    
    # Guardar el resultado
    output_shapefile = "./data/Troncales_Unidas.shp"
    output_geojson = "./data/Troncales_Unidas.geojson"
    
    #output_shapefile = os.path.join(root,'cars_filtered_unidas.shp')
    #output_geojson = output_shapefile.replace('.shp','.geojson')
    
    merged_gdf.to_file(output_shapefile)
    #merged_gdf.to_file(output_geojson, driver="GeoJSON")
    
    print(f"Archivos combinados guardados en:")
    print(f"Shapefile: {output_shapefile}")
    #print(f"GeoJSON: {output_geojson}")
else:
    print("No se encontraron archivos válidos para combinar.")

Archivo cargado: D:\CORREDORES_V_DATASET\MASKS_VIAS_FILL_Z19_BB22\Troncal1.geojson
Archivo cargado: D:\CORREDORES_V_DATASET\MASKS_VIAS_FILL_Z19_BB22\Troncal2.geojson
Archivo cargado: D:\CORREDORES_V_DATASET\MASKS_VIAS_FILL_Z19_BB22\Troncal3.geojson
Archivo cargado: D:\CORREDORES_V_DATASET\MASKS_VIAS_FILL_Z19_BB22\Troncal4.geojson
Archivo cargado: D:\CORREDORES_V_DATASET\MASKS_VIAS_FILL_Z19_BB22\Troncal5.geojson
Archivo cargado: D:\CORREDORES_V_DATASET\MASKS_VIAS_FILL_Z19_BB22\Troncal9.geojson
Archivos combinados guardados en:
Shapefile: D:\CORREDORES_V_DATASET\Troncales_Unidas.shp


# Unir vías y locations

In [ ]:

shp_vias="./data/Troncales_Unidas.shp"
gdf_v=gpd.read_file(shp_vias)
gdf_v

csv_loc=r'C:\Users\Sebastian\Documents\CORREDORES_V\locations2.csv'

df=pd.read_csv(csv_loc)

# 1. Creas en df una columna con el nombre recortado antes del primer punto
gdf_v['id'] = gdf_v['origen'].str.split('.', n=1).str[0]
df['id'] = df['Archivo'].str.split('.', n=1).str[0]

# 2. Haces el merge usando left_on=gdf["origen"] y right_on=df["Archivo_sin_extension"]
merged = gdf_v.merge(df, on='id', how='left')

# De esta forma, merged tendrá todas las columnas de gdf y también las de df.
# Recuerda que si quieres que siga siendo un GeoDataFrame, puedes forzar:
merged = gpd.GeoDataFrame(merged, geometry='geometry')
merged = merged.drop(columns=['FID', 'origen', 'Archivo'])

shp_vias_loc=shp_vias.replace('.shp','_loc.shp')
merged.to_file(shp_vias_loc)

excel_path = shp_vias_loc.replace('.shp', '.xlsx')
df_excel = merged.copy()
df_excel['geometry'] = df_excel['geometry'].apply(lambda geom: geom.wkt)  # Convertir geometría a WKT (texto)
df_excel.to_excel(excel_path, index=False)

C:\Users\Sebastian\AppData\Local\Temp\ipykernel_12356\1785146483.py:22: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  merged.to_file(shp_vias_loc)
c:\Users\Sebastian\anaconda3\envs\ds_311\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'municipality' to 'municipali'
  ogr_write(
c:\Users\Sebastian\anaconda3\envs\ds_311\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'ISO3166-2-lvl4' to 'ISO3166-2-'
  ogr_write(
c:\Users\Sebastian\anaconda3\envs\ds_311\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'country_code' to 'country_co'
  ogr_write(
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_12356\1785146483.py:26: UserWarning: Geometry column does not contain geometry.
  df_excel['geometry'] = df_excel['geometry'].apply(lambda geom: geom.wkt)  # Convertir geometría a WKT (texto)


# Unir vías y carros

In [ ]:
import os,glob
import geopandas as gpd

shp_cars='./data/IMAGES\SAMprompts_ONLY_ROADS_FILL_z21_ExtBB22\cars, buses and trucks\cars_filtered_unidas.shp'
shp_vias_loc=r'D:\\CORREDORES_V_DATASET\\Troncales_Unidas_loc.shp'

gdf_c=gpd.read_file(shp_cars)
gdf_v=gpd.read_file(shp_vias_loc)

print(gdf_c.columns,gdf_c.shape)
print(gdf_v.columns)


Index(['value', 'origen', 'area_m2', 'geometry'], dtype='object') (6196, 4)
Index(['id', 'Name', 'lat', 'lon', 'road', 'suburb', 'village', 'municipali',
       'county', 'state', 'ISO3166-2-', 'postcode', 'country', 'country_co',
       'geometry'],
      dtype='object')


In [ ]:
#Conteo de carros
print(gdf_c.shape)
gdf_c_agg = gdf_c.dissolve(by='origen')  
print(gdf_c_agg.shape)
# 2. Crear una serie con el conteo de filas por cada origen
conteo_series = gdf_c.groupby('origen').size().rename('conteo_cars')

# 3. Unir ese conteo a gdf_agg
gdf_c_agg = gdf_c_agg.join(conteo_series)

# 4. Opcional: si necesitas que 'origen' pase de índice a columna, haz reset_index()
gdf_c_agg = gdf_c_agg.reset_index()
gdf_c_agg['id'] = gdf_c_agg['origen'].str.split('.', n=1).str[0]

print(gdf_c_agg.shape)

shp_cars_count=shp_vias_loc.replace('.shp','_agg_count.shp')
xlsx_cars_count=shp_cars_count.replace('.shp','.xlsx')

gdf_c_agg.to_file(shp_cars_count)

# 8. Exportar a Excel (con geometría en WKT como texto legible)
df_excel = gdf_c_agg.copy()
df_excel['geometry'] = df_excel['geometry'].apply(lambda g: g.wkt)
df_excel.to_excel(xlsx_cars_count, index=False)

print(f"Exportado correctamente a:\n- {shp_cars_count}\n- {xlsx_cars_count}")

(6196, 4)
(4529, 3)
(4529, 6)


C:\Users\Sebastian\AppData\Local\Temp\ipykernel_12356\3893229220.py:20: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_c_agg.to_file(shp_cars_count)
c:\Users\Sebastian\anaconda3\envs\ds_311\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'conteo_cars' to 'conteo_car'
  ogr_write(
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_12356\3893229220.py:24: UserWarning: Geometry column does not contain geometry.
  df_excel['geometry'] = df_excel['geometry'].apply(lambda g: g.wkt)


Exportado correctamente a:
- D:\\CORREDORES_V_DATASET\\Troncales_Unidas_loc_agg_count.shp
- D:\\CORREDORES_V_DATASET\\Troncales_Unidas_loc_agg_count.xlsx


In [ ]:
import geopandas as gpd
import pandas as pd

# Cargar ambos shapefiles
shp1_path = './data/Troncales_Unidas_loc.shp'  # Shapefile vias
shp2_path = './data/Troncales_Unidas_loc_agg_count.shp'  # Shapefile cars

gdf_shp1 = gpd.read_file(shp1_path)
gdf_shp2 = gpd.read_file(shp2_path)

# Asegurar que la columna 'id' es del mismo tipo en ambos GeoDataFrames
gdf_shp1['id'] = gdf_shp1['id'].astype(str)
gdf_shp2['id'] = gdf_shp2['id'].astype(str)

# Asegurar que ambos GeoDataFrames tengan el mismo CRS
if gdf_shp1.crs != gdf_shp2.crs:
    gdf_shp2 = gdf_shp2.to_crs(gdf_shp1.crs)

# Realizar la unión de los GeoDataFrames usando la columna 'id'
gdf_unido = gdf_shp1.merge(gdf_shp2, on='id', how='left')

# Identificar las columnas de geometría
geom_columns = gdf_unido.columns[gdf_unido.dtypes == 'geometry'].tolist()

# Mantener solo la geometría original del shapefile principal (gdf_shp1)
if 'geometry_x' in gdf_unido.columns and 'geometry_y' in gdf_unido.columns:
    gdf_unido = gdf_unido.drop(columns=['geometry_y'])
    gdf_unido = gdf_unido.rename(columns={'geometry_x': 'geometry'})

# Asegurarse que GeoDataFrame tiene solo una geometría válida
gdf_unido = gpd.GeoDataFrame(gdf_unido, geometry='geometry')

# Exportar el resultado como Shapefile
output_shp = shp2_path.replace('.shp','_cars_vias.shp')
gdf_unido.to_file(output_shp, driver='ESRI Shapefile')

# Exportar el resultado como Excel
output_excel = output_shp.replace('.shp','.xlsx')
gdf_unido.to_excel(output_excel, index=False)

print("✅ Proceso completado exitosamente. Se generaron un Shapefile y un Excel consolidados.")


✅ Proceso completado exitosamente. Se generaron un Shapefile y un Excel consolidados.


In [ ]:
#union de todods los resultados incluyendo anomalies

import geopandas as gpd
import pandas as pd

# Cargar el shapefile en un GeoDataFrame
shp_path = r'D:\\CORREDORES_V_DATASET\\Troncales_Unidas_loc_agg_count_cars_vias.shp'  # Reemplaza con la ruta a tu shapefile
gdf_shp = gpd.read_file(shp_path)

# Cargar el archivo Excel en un DataFrame
excel_path = './data/Troncales_anomalies.xlsx'  # Reemplaza con la ruta a tu archivo Excel
df_excel = pd.read_excel(excel_path)

# Asegurar que la columna 'id' es del mismo tipo en ambos DataFrames
gdf_shp['id'] = gdf_shp['id'].astype(str)
df_excel['id'] = df_excel['id'].astype(str)

# Realizar el merge usando la columna 'id'
gdf_unido = gdf_shp.merge(df_excel, on='id', how='left')

# Asegurarse que el resultado es un GeoDataFrame con la geometría adecuada
gdf_unido = gpd.GeoDataFrame(gdf_unido, geometry='geometry')
# Crear la columna 'anomalies' en gdf_unido
gdf_unido['anomalies'] = gdf_unido['anomalies_pct'].apply(lambda x: 1 if (pd.notna(x) and x > 0) else 0)
gdf_unido['cars'] = gdf_unido['conteo_car'].apply(lambda x: 1 if (pd.notna(x) and x > 0) else 0)
gdf_unido['troncal'] = gdf_unido['id'].astype(str).str.split('_').str[0]


columnas_finales=['troncal','id', 'lat', 'lon', 'road', 'suburb', 'village', 'municipali',
       'county', 'state', 'country', #'ISO3166-2-', 'postcode', 'country_co',
       'conteo_car', 'cars', 'anomalies_pct', 'anomalies', 'geometry']

gdf_unido[columnas_finales]

# Exportar el resultado como un nuevo Shapefile
output_shp = './data/shapefile_vias_cars_anomalies.shp'
gdf_unido.to_file(output_shp, driver='ESRI Shapefile')

# Exportar el resultado como un nuevo Excel
output_excel = output_shp.replace('.shp','.xlsx')
gdf_unido.to_excel(output_excel, index=False)

print("✅ Proceso completado exitosamente. Se generaron un Shapefile y un Excel consolidados.")


C:\Users\Sebastian\AppData\Local\Temp\ipykernel_21720\1244955340.py:37: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_unido.to_file(output_shp, driver='ESRI Shapefile')
c:\Users\Sebastian\anaconda3\envs\ds_311\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Unnamed: 0' to 'Unnamed_ 0'
  ogr_write(
c:\Users\Sebastian\anaconda3\envs\ds_311\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'anomalies_pct' to 'anomalies_'
  ogr_write(


✅ Proceso completado exitosamente. Se generaron un Shapefile y un Excel consolidados.


In [12]:
gdf_unido.anomalies.value_counts()

anomalies
0    37505
1      336
Name: count, dtype: int64